# Sinhala MMLU — zero-shot log-prob evaluation (Colab)
**Runtime:** T4 GPU. **Secrets:** `HF_TOKEN` (notebook access ON); accept the Gemma-3 and Llama-3.2 licences on HF first.
Everything is read from and written to Drive, so results survive session resets.

**Eval set = public train (Easy only, never trained on). Public test is used only to write predictions (section 6).**
Run sections 1-5 top to bottom. If Colab disconnects, run section 1 again, then the grid (section 4): it skips finished experiments.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Sinhala MMLU"
!ls

In [ ]:
!pip -q install -r requirements-colab.txt
import torch, transformers, sys
print(sys.version.split()[0], torch.__version__, transformers.__version__, torch.version.cuda, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!pip freeze > requirements-lock.txt   # exact versions this run used

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['PYTHONHASHSEED'] = '0'
!python -X utf8 tests/test_eval_core.py
!python -X utf8 src/validate_data.py
!python -X utf8 src/build_dev.py

## 2. Fresh start (OPTIONAL, run only when you want to throw away ALL previous results)
Deletes everything in `outputs/`, `results/subject_accuracy/`, `results/invalid_rows.csv`, and empties `results/results.csv` (header kept). Set `CONFIRM = True` to run it. Never run it while a grid is running.

In [ ]:
CONFIRM = False   # <- set to True to wipe all previous results
import shutil, pathlib
root = pathlib.Path('.').resolve()
if CONFIRM:
    for p in (root / 'outputs').glob('*'):
        shutil.rmtree(p) if p.is_dir() else p.unlink()
    shutil.rmtree(root / 'results' / 'subject_accuracy', ignore_errors=True)
    (root / 'results' / 'invalid_rows.csv').unlink(missing_ok=True)
    header = 'exp_id,date,model,method,prompt,dev_easy,ci95,digit_mass,notes'
    (root / 'results' / 'results.csv').write_text(header + chr(10))
    print('wiped')
print('outputs:', sorted(p.name for p in (root / 'outputs').glob('*')))
print((root / 'results' / 'results.csv').read_text())

## 3. Checks before the grid
**3a. Token-length histogram** (tokenizer only, no model weights, no GPU work): one histogram per model for the 1,851 eval prompts, exactly as they are fed to the model (chat template included for chat-capable models). Saved to `results/figures/`. Prompts are never truncated: a prompt longer than the model's own context length raises an error.
**3b. Smoke test** (20 questions; results go to a `_smoke` folder and are not added to `results.csv`). Check `[run] chat=...`, `digit_token_ids` and `max_token_len`.

In [ ]:
# Which models to run (space-separated config files). Now: Qwen only.
# All three: 'configs/qwen3_0.6b.yaml configs/gemma3_1b.yaml configs/llama32_1b.yaml'
CONFIGS = 'configs/qwen3_0.6b.yaml'

In [ ]:
!python -X utf8 src/token_hist.py --configs {CONFIGS}
from IPython.display import Image, display
display(Image('results/figures/token_hist_dev_easy_full.png'))

In [ ]:
!python -X utf8 src/eval_logprob.py --config configs/qwen3_0.6b.yaml --data dev/dev_easy_screen.jsonl --limit 20 --chat on
!python -X utf8 src/eval_logprob.py --config configs/qwen3_0.6b.yaml --data dev/dev_easy_screen.jsonl --limit 20 --chat off

## 4. Experiment grid (full precision, full Easy dev, 1,851 questions)
Per model: chat on/off x {calibration off + 1 option order, calibration off + 4 option orders, calibration on + 1 option order} = 6 experiments (18 in total). Models are loaded once. Finished experiments are skipped, so this cell can be re-run after a disconnect. Subject-level CSVs are refreshed after every model.

**Prompt:** `nosubject` (no subject name in the prompt), set in `configs/*.yaml`. To switch for a comparison run `src/run_grid.py --prompt subject_si` (results are kept separately: the prompt is part of every experiment name).

**Validity check:** `digit_mass` in `results.csv` = average total probability the model puts on the four answer digits (1.0 = it always wants to answer with a digit). Below 0.5 the run prints a WARNING: the argmax over the digits is then weak evidence.

In [ ]:
!python -X utf8 src/run_grid.py --dry-run --configs {CONFIGS}

In [ ]:
!python -X utf8 src/run_grid.py --configs {CONFIGS}

## 5. Results
`results/results.csv` has one row per experiment (overall Easy accuracy, 95% interval and digit_mass). `results/subject_accuracy/<Model>.csv` has accuracy per subject (JSON file) for each experiment.

In [ ]:
import pandas as pd, glob
display(pd.read_csv('results/results.csv')[['exp_id','model','method','dev_easy','ci95','notes']])
!python -X utf8 src/subject_report.py
for f in sorted(glob.glob('results/subject_accuracy/*.csv')):
    print(f); display(pd.read_csv(f))

## 6. Public-test predictions (only after the dev results are decided)
No diagnostics are printed and test predictions must never be inspected. Output: `outputs/<exp_id>/predictions.zip` (for Codabench). Example: the Gemma-3-1B format test.

In [ ]:
!python -X utf8 src/eval_logprob.py --config configs/gemma3_1b.yaml --data test --chat on --exp-id gemma3_1b_chaton